Neural Network/ Vanilla Perceptrons/ Artificial Neural Networs:

Where perceptron fails? And need for multi-layer Perceptrons:

![My Image](Images/Neural%20Network%20architecture.png)


In [1]:
import numpy as np

In [2]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

In [14]:
def sigmoid_gradient(sigmoid):
    return np.multiply(sigmoid, (1-sigmoid))

Here is the mathematical formula of the softmax:


$$
\text{softmax}(l_i) = \frac{e^{l_i}}{\sum e^{l}}
$$
 

In [13]:
def softmax(logits):
    exponentials = np.exp(logits)
    return exponentials / np.sum(exponentials,axis=1).reshape(-1,1)

logits is the fancy name for the inputs of the softmax. In our case, those inputs are arranged in a matrix that has one row for each MNIST image and one column per class. For example, if we train the system with 60,000 images, then softmax() gets a (60000,10) matrix, and it must return a matrix of the same shape.

The first line in softmax() calculates the exponentials of the logits. The second line divides each exponential by the sum of all the exponentials on the same row. The parameter axis=1. Means to calculate the sum by row, not over the entire matrix. Before doing the division, we must reshape the sums into a one-column matrix. Otherwise, NumPy outputs an error saying that it cannot divide a matrix by a one-dimensional array.

Right now we are using Sigmoid function which can either overflow or underflow when class values are too high or too low. And function will end up returning "not a number" outputs. Bad for model...

To resolve this we can use stable implementation of softmax, that avoid extreme
intermediate values.

$$
\text{softmax}(l_i) = \frac{e^{l_i} + max(e^{l})}{\sum e^{l} + max(e^{l})}
$$

Try with softmax(np.array([[1, 1000]]))

In [3]:
def prepend_bias(X):
    return np.insert(X,0,1,axis=1)

In [15]:
def forward(X,w1,w2):
    h = sigmoid(np.matmul(prepend_bias(X),w1))
    y_hat = softmax(np.matmul(prepend_bias(h),w2))
    return (y_hat,h)

Cross Entropy Loss

While the log loss served us well so far, it’s time to switch to a simpler formula, one that’s specific to multiclass classifiers. It’s called the cross-entropy loss, and it looks like this:

$$
L = -\frac{1}{m} \sum y_{i} \log(\hat{y}_{i})
$$

A pragmatic reason to use the cross-entropy loss in our neural network is that it is a perfect match for the softmax. More specifically, a softmax followed by a cross-entropy loss makes it easier to code gradient descent. For now, just know that the softmax and the cross-entropy loss work well together

we don’t care about the loss as much as the gradient of the loss, which we’ll use later during gradient descent. While the loss() function is not really necessary, however, it is still useful. We can look at that number to gauge how well the classifier is doing, both during training and during classification. That’s the reason why we code loss() and call it from the report() function.

In [17]:
def loss(Y,y_hat):
    return -np.sum(Y * np.log(y_hat))/Y.shape[0]

The revenge of local minima

While talking about loss functions, we must note that because When Gradient Descent Fails, we vetted our loss functions for a major requirement. They had to play well with gradient descent. In particular, the GD algorithm can get stuck in a local minimum—so we picked loss functions that did not have local minima.

Unfortunately, once we graduate to neural networks, all bets are off. However we select it carefully, the loss of a neural network can have local minima. Neural networks have more sophisticated models than perceptrons that allow them to approximate more complicated functions.As a downside, the losses of those functions can have holes, where GD gets stuck.

On the bright side, even though a neural network’s loss can have local minima, that does not mean that it always does—and even when local minima do exist, they are not necessarily a showstopper. As researchers understand more about neural networks, they find out that local minima are not quite as disruptive as we assumed. Most of the time, when we train a neural network, we’ll find a minimum that’s good enough, even though we cannot be sure it is the best minimum overall.

The need of Backpropogation

So far we develope neural network that takes input and weights and provide label output. But how can we find those weights?

Thats' where we need backprop => an algorithm that calculates the gradients of the loss in any neural network.

In the case of the perceptron, we calculated the derivatives of the loss with respect to the weights to get the gradient. However, in the case of a neural network, coming up with the equivalent derivatives can be hard. For example, the code that computes the loss in our three-layered network is given below:

h = sigmoid(matmul(X, w1))
y_hat = softmax(matmul(h, w2))
L = cross_entropy_loss(Y, y_hat)


Just for  one hidden layer we need hefty work. taking derivatives of that formula with respect to w1 and w2. Morden nueral netowrks are more complex and it could be more chllenging. back prop make it easy.

From the Chain Rule to Backpropogation

Backprop is application of chain rule. The chain rule calculates the gradient of any node y with respect to any other node x, we multiply the local gradient of all the nodes on the way back from y to x.

![My Image](Images/Chain%20Rule%20in%20NN.png)

Assume x=3,y=17,w1=6 and w2=2 at given point of time while training.

To take a step of GD, we need the gradients of L with respect to 
w1 and w2. We can compute those gradients with the chain rule. 
∂L/∂w is the product of the local gradients on the way back from 
L to w1. The same goes for ∂L/∂w2.

"the gradient of w1" or "the gradient of L with respct to w1" is:

$$
\frac{∂L}{∂w_1} = \frac{∂L}{∂\hat{y}} * \frac{∂\hat{y}}{∂w_1} = -6 * 1 = -6
$$

With the current weights, the gradient of w1 is -6. To take the next step of GD, we multiply this gradient by the learning rate, and subtract the result from w1.

We can follow a similar process to get 
∂L/∂w2, although in this case we have an additional complication: there are two paths leading from w2 to L — one that passes by the multiplication, and one that does not. Whenever we have multiple paths, we have to sum their gradients:

$$
\frac{∂L}{∂w_2} = \frac{∂L}{∂\hat{y}} * \frac{∂\hat{y}}{∂w_2}  +  \frac{∂L}{∂\hat{y}} * \frac{∂\hat{y}}{∂a} * \frac{∂a}{∂w_2} = (-6 * 1) + (-6*1*3)= -24
$$

∂L/∂w2 is also negative, but larger than ∂L/∂w1. Once again, we can multiply this gradient by the learning rate, and subtract the result from w2.

n general, if a weight has a small gradient, that means that it does not contribute much to the network’s error, so it can change just a little bit. Conversely, a weight with a large gradient has a big impact on the network’s error and needs to be changed more decisively. Backprop is a way to calculate how much each weight needs to change.

Backpropagation moves back from the loss to the weights, accumulating local gradients through the chain rule.

Calculating Gradient for out Network:

![My Image](Images/Backprop%20for%20NN.png)

we established a fact that the softmax and the cross-entropy loss are a perfect match. Now we can see why. Taken separately, those functions have complicated derivatives. When we compose them, their derivatives simplify to a formula: the network’s output minus the ground truth. This formula is short and simple to compute.

Gradient of w2:

$$
\frac{∂L}{∂w_2} = {SML'} * \frac{∂b}{∂w_2} = (\hat{y} - y)*h
$$

In [18]:
#w2_gradient = np.matmul(prepend_bias(h).T, y_hat-Y) / X.shape[0]

Gradient of w1:

$$
\frac{∂L}{∂w_1} = {SML'} * \frac{∂b}{∂h} * σ' * \frac{∂a}{∂w_1} = (\hat{y} - y) * w_2 * σ * (1-σ)*x
$$

here σ' = ∂h/∂a

In [19]:
def sigmoid_gradient(sigmoid):
    return np.multiply(sigmoid, (1-sigmoid))

#a_gradient = np.matmul(y_hat - Y, w2[1:].T) * sigmoid_gradient(h)
#w1_gradient = np.matmul(prepend_bias(X).T, a_gradient) / X.shape[0]

The second last line calculates (y_hat−y)⋅w_2⋅σ′. There are a few complexities involved in this calculation. To begin with, this time around we use h as it is without a bias column. Fact is, the bias column gets added after the calculation of h, so its gradient does not propagate as far back as the gradient of 
w1. In other words, that column has no effect on ∂L/∂w1.

Now, because we ignored the first column of h, we also have to ignore its weights. That’s the first row of w2, because matrix multiplication matches columns by rows. So, the w2[1:] means w2, without the first row.

In the next line of code, the sigmoid_gradient() helper function calculates the sigmoid’s gradient from the sigmoid’s output. We already know that the sigmoid’s output is the hidden layer h, so we can just call sigmoid_gradient(h).

The last line in the code finishes the job, it multiplies the previous intermediate result by X. This multiplication involves the same process when we calculated the gradient of w2: we need to swap the operands, transpose one of the matrices, prepend the bias column to X like we do during forward propagation, and average the final gradient over the training examples.

In [20]:
#Combined backprop code

def back(X,Y,y_hat,w2,h):
    w2_gradient = np.matmul(prepend_bias(h).T,(y_hat - Y)) / X.shape[0]
    w1_gradient = np.matmul(prepend_bias(X).T, np.matmul(y_hat-Y,w2[1:].T)
                            * sigmoid_gradient(h))/X.shape[0]
    
    return (w1_gradient,w2_gradient)

In [21]:
def classify(X,w1,w2):
    y_hat,_ = forward(X,w1,w2)
    labels = np.argmax(y_hat,axis=1)
    return labels.reshape(-1,1)

Initialize the Weights:

Checkout experiment done in directory 5.1 on initialize all weights to same value and larger weights to understand why it cause problem.

So, how small weights should be?

$$
w \approx \pm \sqrt{\frac{1}{r}}
$$

r = number of rows in weight matix


In [22]:
def initialize_weights(n_input_variables, n_hidden_nodes,n_classes):
    w1_rows = n_input_variables + 1
    w1 = np.random.randn(w1_rows,n_hidden_nodes) * np.sqrt(1/w1_rows)

    w2_rows = n_hidden_nodes + 1
    w2 = np.random.randn(w2_rows,n_classes) * np.sqrt(1/w2_rows)

    return (w1,w2)

In [23]:
def report(iteration,X_train,Y_train,X_test,Y_test,w1,w2):
    y_hat,_ = forward(X_train,w1,w2)
    training_loss  = loss(Y_train,y_hat)
    classifications = classify(X_test,w1,w2)
    accuracy = np.average(classifications == Y_test) * 100.0
    print("Iteration: %5d, Loss: %.6f, Accuracy: %.f%%" % 
          (iteration,training_loss,accuracy))


In [24]:
#training phase

def train(X_train,Y_train,X_test,Y_test,n_hidden_nodes,iterations,lr):
    n_input_variables = X_train.shape[1]
    n_classes = Y_train.shape[1]
    w1,w2 = initialize_weights(n_input_variables,n_hidden_nodes,n_classes)
    for iteration in range(iterations):
        y_hat, h = forward(X_train,w1,w2)
        w1_gradient, w2_gradient = back(X_train,Y_train,y_hat,w2,h)
        w1-=(w1_gradient * lr)
        w2-=(w2_gradient * lr)
        report(iteration,X_train,Y_train,X_test,Y_test,w1,w2)
    return (w1,w2)

In [25]:
# An MNIST loader.

import numpy as np
import struct


def load_images(filename):
    # Open and unzip the file of images:
    with open(filename, 'rb') as f:
        # Read the header information into a bunch of variables:
        _ignored, n_images, columns, rows = struct.unpack('>IIII', f.read(16))
        # Read all the pixels into a NumPy array of bytes:
        all_pixels = np.frombuffer(f.read(), dtype=np.uint8)
        # Reshape the pixels into a matrix where each line is an image:
        return all_pixels.reshape(n_images, columns * rows)


# 60000 images, each 784 elements (28 * 28 pixels)
X_train = load_images("Datasets/archive/train-images-idx3-ubyte/train-images-idx3-ubyte")

# 10000 images, each 784 elements, with the same structure as X_train
X_test = load_images("Datasets/archive/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte")


def load_labels(filename):
    # Open and unzip the file of images:
    with open(filename, 'rb') as f:
        # Skip the header bytes:
        f.read(8)
        # Read all the labels into a list:
        all_labels = f.read()
        # Reshape the list of labels into a one-column matrix:
        return np.frombuffer(all_labels, dtype=np.uint8).reshape(-1, 1)


def one_hot_encode(Y):
    n_labels = Y.shape[0]
    n_classes = 10
    encoded_Y = np.zeros((n_labels, n_classes))
    for i in range(n_labels):
        label = Y[i]
        encoded_Y[i][label] = 1
    return encoded_Y

# 60K labels, each with single digit from 0 to 9
Y_train_unencoded = load_labels("Datasets/archive/train-labels-idx1-ubyte/train-labels-idx1-ubyte")

# 60K labels, each consisting of 10 one-hot encoded elements
Y_train = one_hot_encode(Y_train_unencoded)

# 10000 labels, each a single digit from 0 to 9
Y_test = load_labels("Datasets/archive/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte")




In [28]:
w1, w2 = train(X_train, Y_train,
               X_test, Y_test,
               n_hidden_nodes=200, iterations=1000, lr=0.01)

Iteration:     0, Loss: 2.595681, Accuracy: 10%
Iteration:     1, Loss: 2.514976, Accuracy: 11%
Iteration:     2, Loss: 2.452197, Accuracy: 12%
Iteration:     3, Loss: 2.400126, Accuracy: 13%
Iteration:     4, Loss: 2.355626, Accuracy: 14%
Iteration:     5, Loss: 2.315806, Accuracy: 15%
Iteration:     6, Loss: 2.279674, Accuracy: 17%
Iteration:     7, Loss: 2.246543, Accuracy: 18%
Iteration:     8, Loss: 2.215680, Accuracy: 20%
Iteration:     9, Loss: 2.186630, Accuracy: 21%
Iteration:    10, Loss: 2.159053, Accuracy: 23%
Iteration:    11, Loss: 2.132705, Accuracy: 25%
Iteration:    12, Loss: 2.107430, Accuracy: 27%
Iteration:    13, Loss: 2.082975, Accuracy: 29%
Iteration:    14, Loss: 2.059240, Accuracy: 31%
Iteration:    15, Loss: 2.035929, Accuracy: 33%
Iteration:    16, Loss: 2.013101, Accuracy: 35%
Iteration:    17, Loss: 1.990905, Accuracy: 37%
Iteration:    18, Loss: 1.969173, Accuracy: 39%
Iteration:    19, Loss: 1.948016, Accuracy: 41%
Iteration:    20, Loss: 1.927634, Accura

KeyboardInterrupt: 